# Boss Monster on OpenSpiel — play over ngrok

This notebook starts a 2-player [Boss Monster](https://boardgamegeek.com/boardgame/131835/boss-monster-the-dungeon-building-card-game)
web server (an OpenSpiel Python game, `python_boss_monster`) and exposes it
to the internet through an [ngrok](https://ngrok.com) tunnel, so two people
anywhere can play against each other from their own browsers.

**Before running:** get a free ngrok account and copy your authtoken from
https://dashboard.ngrok.com/get-started/your-authtoken — you'll paste it
into the cell below.

**How it works:** we `pip install open_spiel` to get the compiled `pyspiel`
core fast (building OpenSpiel from source in Colab is slow), then `git
clone` just the repo containing the Boss Monster game/server source (which
is Python-only and isn't part of the PyPI wheel yet) and put it earlier on
`sys.path` so it's used instead of the wheel's copy. Everything else
(the C++ engine, NumPy, etc.) still comes from the pip package.

## 1. Install dependencies

In [ ]:
!pip install -q open_spiel flask pyngrok

## 2. Get the Boss Monster source

Edit `REPO_URL` / `REPO_BRANCH` if you're working from your own fork or
branch.

In [ ]:
REPO_URL = "https://github.com/calvinpozderac-claude/open_spiel.git"
REPO_BRANCH = "claude/boss-monster-6j6191"

import os
if not os.path.isdir("open_spiel_src"):
  !git clone --depth 1 --branch $REPO_BRANCH $REPO_URL open_spiel_src
else:
  !cd open_spiel_src && git fetch --depth 1 origin $REPO_BRANCH && git checkout $REPO_BRANCH && git reset --hard origin/$REPO_BRANCH

import sys
SRC = os.path.abspath("open_spiel_src")
if SRC not in sys.path:
  sys.path.insert(0, SRC)
print("Using OpenSpiel python source from:", SRC)

In [ ]:
# Sanity check: this should point inside open_spiel_src, not site-packages.
from open_spiel.python.games import boss_monster
print(boss_monster.__file__)

import pyspiel
game = pyspiel.load_game("python_boss_monster")
print(game)
state = game.new_initial_state()
print("New game created OK. Current player:", state.current_player())

## 3. Set your ngrok authtoken

In [ ]:
import getpass
NGROK_AUTHTOKEN = getpass.getpass("Paste your ngrok authtoken: ")

## 4. Start the server and open the tunnel

Re-run this cell any time you want a fresh public URL (e.g. after an ngrok
session expires on the free tier).

In [ ]:
from open_spiel.python.examples import boss_monster_ngrok

tunnel = boss_monster_ngrok.run(port=8080, ngrok_authtoken=NGROK_AUTHTOKEN)
print("\nOpen this URL, click 'New Game', and send the two /play/<token>")
print("links it shows you to your two players:\n")
print(tunnel.public_url)

## 5. (Optional) Shut the tunnel down

In [ ]:
from pyngrok import ngrok
ngrok.disconnect(tunnel.public_url)
# ngrok.kill()  # uncomment to also stop the local ngrok agent process

## Appendix: run OpenSpiel's own tests on the game

Confirms the game passes OpenSpiel's API-conformance checks
(`pyspiel.random_sim_test`) in this environment.

In [ ]:
import pyspiel
pyspiel.random_sim_test(game, num_sims=5, serialize=True, verbose=False)
print("OK")